In [ ]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly joblib scikit-learn pandas numpy transformers accelerate bitsandbytes kaggle

In [ ]:
!pip show streamlit

In [ ]:
!which streamlit

In [ ]:
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "freightquote_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FreightQuote_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")
except Exception as e:
    STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")

os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
print(f"📁 Storage: {STORAGE_DIR}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")


In [ ]:
import os

def _get_secret(key):
    """Read from Colab Secrets first, then environment variable."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

# ── Load all 7 secrets (set these in Colab Secrets panel) ──────────────────
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "freightquote_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

# Expose Kaggle credentials for the kaggle library
if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

# ── Mount Google Drive (auto-detected in Colab) ─────────────────────────────
try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FreightQuote_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")
except Exception as e:
    print(f"⚠️  Drive mount skipped ({e}). Using local storage.")
    STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")

os.makedirs(STORAGE_DIR, exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)

print(f"\n📁 Storage:  {STORAGE_DIR}")
print(f"🔑 JWT:      {'✅ from Colab Secrets' if _get_secret('JWT_SECRET_KEY') else '⚠️  using dev default'}")
print(f"🔑 Admin:    {ADMIN_EMAIL}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Kaggle:   {'✅' if KAGGLE_KEY else '❌ optional — synthetic fallback'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Email:    {'✅' if EMAIL_PASSWORD else '❌ optional'}")


In [ ]:
!nvidia-smi

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
print("✅ Qwen-2.5-3B loaded. Footprint (GB):", round(model.get_memory_footprint() / 1e9, 2))


In [ ]:
%%writefile app.py
import os, re, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, streamlit as st
import joblib, numpy as np, pandas as pd
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

from db import get_conn as agent_db_conn, load_chat_history, save_chat_message, clear_chat_history
from agent2_freight import render_agent2_freight
from agent3_freight import render_agent3_freight
from llm_engine_freight import (orchestrate_3_agents_query, generate_debate_and_synthesis,
                                 warmup_llm, is_llm_loaded, start_background_warmup)
from config import AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT3_MODEL_PATH

start_background_warmup()

# ============================================================
# 0. PAGE CONFIG + FORCE LIGHT THEME
# ============================================================
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#3aa0ff"\nbackgroundColor="#ffffff"\n'
            'secondaryBackgroundColor="#eaf4fd"\ntextColor="#123a5e"\n')

st.set_page_config(page_title="Freight Quote", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

# ============================================================
# 1. LIGHT-BLUE & WHITE COLOR PALETTE
# ============================================================
COLORS = {
    "bg_main": "#ffffff", "bg_sidebar": "#eaf4fd", "bg_card": "#ffffff", "bg_card_alt": "#d6ecff",
    "text_main": "#123a5e", "text_heading": "#0b2942", "text_muted": "#5b7a99",
    "accent": "#3aa0ff", "accent_hover": "#1e86e6", "accent_text": "#ffffff",
    "border": "#0b2942", "border_light": "#bfe2ff", "success": "#34d399", "danger": "#f87171"
}

# ============================================================
# 2. SECRETS
# ============================================================
JWT_SECRET = os.environ.get("JWT_SECRET")
SENDER_EMAIL = os.environ.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")
ADMIN_EMAIL = os.environ.get("ADMIN_EMAIL", "admin@freightquote.com")
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD")
OTP_EXPIRY_MINUTES = 5

if not JWT_SECRET:
    st.error("❌ JWT_SECRET is not set. Add it to Colab Secrets.")
    st.stop()

# ============================================================
# 3. CSS
# ============================================================
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: #ffffff !important; color: #ffffff !important; stroke: #ffffff !important;
    }}
    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
    div[data-testid="stVerticalBlockBorderWrapper"] {{
        background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 16px !important; padding: 24px !important; box-shadow: 6px 6px 0px {COLORS['border_light']} !important;
    }}
</style>
""", unsafe_allow_html=True)

# ============================================================
# 4. DATABASE
# ============================================================
def get_db(): return sqlite3.connect("freightquote_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    for col, ddl in [
        ("role", "TEXT DEFAULT 'Customer'"),
        ("failed_attempts", "INTEGER DEFAULT 0"),
        ("lock_until", "TIMESTAMP"),
        ("account_status", "TEXT DEFAULT 'active'"),
    ]:
        try:
            conn.execute(f"ALTER TABLE users ADD COLUMN {col} {ddl}")
        except Exception:
            pass
    if ADMIN_PASSWORD and not conn.execute(
        "SELECT 1 FROM users WHERE email=?", (ADMIN_EMAIL,)).fetchone():
        conn.execute(
            "INSERT INTO users (username,email,password_hash,role) VALUES (?,?,?,?)",
            ("Administrator", ADMIN_EMAIL, hash_txt(ADMIN_PASSWORD), "Admin"))

SECURITY_QUESTIONS = [
    "What is your pet's name?",
    "What is your mother's maiden name?",
    "What is your favourite city?",
]

@st.cache_resource
def load_agents():
    m1 = joblib.load(AGENT1_MODEL_PATH) if os.path.exists(AGENT1_MODEL_PATH) else None
    m2 = joblib.load(AGENT2_MODEL_PATH) if os.path.exists(AGENT2_MODEL_PATH) else None
    m3 = joblib.load(AGENT3_MODEL_PATH) if os.path.exists(AGENT3_MODEL_PATH) else None
    return m1, m2, m3

agent1_m, agent2_m, agent3_m = load_agents()

def confidence_band(model, X_row):
    if model is None:
        return 0.5, 0.42, 0.58
    if hasattr(model, "predict_proba"):
        prob = float(model.predict_proba([X_row])[0][1])
    else:
        prob = float(np.clip(model.predict([X_row])[0], 0, 1))
    z, n = 1.96, 300
    lo = max(0.0, (prob + z**2/(2*n) - z*((prob*(1-prob)+z**2/(4*n))/n)**0.5) / (1+z**2/n))
    hi = min(1.0, (prob + z**2/(2*n) + z*((prob*(1-prob)+z**2/(4*n))/n)**0.5) / (1+z**2/n))
    return prob, lo, hi

a1_ctx = {"base_rate_usd": 18500, "congestion": "High", "fuel_surcharge_pct": 13.5}
a2_ctx = {"dwell_days": 3.8, "canal_queue": True, "delay_risk_pct": 68}
a3_ctx = {"carrier": "Maersk", "punctuality": 0.94, "compliance": "Passed"}

# ============================================================
# 5. VALIDATION HELPERS
# ============================================================
EMAIL_RE = re.compile(r'^[A-Za-z]{2,}[A-Za-z0-9._%+\-]*@[A-Za-z]{2,}[A-Za-z0-9\-]*\.[A-Za-z]{2,}$')

def valid_email(e): return bool(EMAIL_RE.match(e or ""))

def valid_password(p):
    if not p or len(p) < 8: return False
    if not re.search(r'[A-Z]', p): return False
    if not re.search(r'[a-z]', p): return False
    if not re.search(r'[0-9]', p): return False
    if not re.search(r'[^A-Za-z0-9]', p): return False
    return True

PASSWORD_HELP = "Min 8 chars, 1 uppercase, 1 lowercase, 1 number, 1 special symbol."

def password_strength(pw):
    """Milestone 2 tiered password badge."""
    if not pw or len(pw) < 5:
        return "weak", "🔴 Password too weak (minimum 5 characters required)."
    elif len(pw) < 10:
        return "average", "🟡 Average strength (10+ characters recommended for enterprise security)."
    else:
        return "good", "🟢 Good password strength — proceed with bcrypt hashing."

# ============================================================
# 6. JWT / OTP HELPERS
# ============================================================
def make_jwt(email):
    return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)},
                       JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception: return None

def make_otp_token(email, otp):
    payload = {"sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp",
               "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]): return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"

def send_otp_email(to_email, otp):
    if not SENDER_EMAIL or not EMAIL_PASSWORD:
        return False, "EMAIL_ADDRESS / EMAIL_PASSWORD not configured in Colab Secrets."
    msg = MIMEMultipart('alternative')
    msg['From'] = "Freight Quote Support <" + SENDER_EMAIL + ">"
    msg['To'] = to_email
    msg['Subject'] = "Freight Quote - Verification Code"
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()
    msg['Reply-To'] = SENDER_EMAIL

    text_body = "Your verification code for Freight Quote is: " + otp + ". This code will expire in " + str(OTP_EXPIRY_MINUTES) + " minutes. If you did not request this code, please ignore this email."
    html_body = "".join([
        "<html><body style='font-family:Arial,sans-serif;background:#eaf4fd;padding:20px;'>",
        "<div style='max-width:480px;margin:0 auto;background:#ffffff;border:2px solid #0b2942;border-radius:12px;padding:30px;text-align:center;'>",
        "<div style='color:#0b2942;font-size:20px;font-weight:bold;margin-bottom:15px;'>Freight Quote Verification</div>",
        "<div style='color:#123a5e;font-size:15px;margin-bottom:20px;'>We received a request to reset the password for <b>", to_email, "</b>. Use the code below:</div>",
        "<div style='background:#3aa0ff;color:#ffffff;font-size:28px;font-weight:bold;letter-spacing:5px;padding:15px 20px;border:2px solid #0b2942;border-radius:8px;display:inline-block;margin:10px 0;'>", otp, "</div>",
        "<div style='color:#123a5e;font-size:14px;margin-top:15px;'>This code expires in <b>", str(OTP_EXPIRY_MINUTES), " minutes</b>.</div>",
        "</div></body></html>",
    ])

    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(SENDER_EMAIL, EMAIL_PASSWORD)
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e:
        return False, "SMTP Error: " + str(e)

# ============================================================
# 7. SESSION STATE
# ============================================================
for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None),
             ("otp_token", None), ("fp_verified", False),
             ("otp_resend_count", 0), ("otp_next_allowed", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Freight Quote Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Freight Quote</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# 8. PAGE ROUTING — LOGIN / SIGNUP / FORGOT PASSWORD
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:

        # ---------------- LOGIN ----------------
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@freightquote.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                if not email or not pwd:
                    st.error("⚠️ Both fields are required.")
                else:
                    with get_db() as c:
                        r = c.execute(
                            "SELECT password_hash, failed_attempts, lock_until, account_status "
                            "FROM users WHERE email=?", (email,)).fetchone()

                    if not r:
                        st.error("❌ Invalid credentials.")
                    else:
                        phash, fails, lock_until, status = r
                        fails = fails or 0
                        now = datetime.datetime.utcnow()
                        locked_until = datetime.datetime.fromisoformat(lock_until) if lock_until else None

                        if status == "locked":
                            st.error("❌ Account permanently locked due to 5 failed attempts. "
                                     "Only the System Administrator can unlock this account via the Admin Dashboard.")
                        elif locked_until and now < locked_until:
                            mins = int((locked_until - now).total_seconds() // 60) + 1
                            st.error(f"⏳ Account temporarily locked. Try again in ~{mins} min.")
                        elif check_txt(pwd, phash):
                            with get_db() as c:
                                c.execute("UPDATE users SET failed_attempts=0, lock_until=NULL WHERE email=?", (email,))
                            st.session_state.token = make_jwt(email)
                            navigate("Dashboard")
                        else:
                            fails += 1
                            new_lock, msg = None, None
                            if fails == 3:
                                new_lock = (now + datetime.timedelta(minutes=5)).isoformat()
                                msg = "⏳ Account temporarily locked for 5 minutes due to 3 failed attempts."
                            elif fails == 4:
                                new_lock = (now + datetime.timedelta(minutes=15)).isoformat()
                                msg = "⏳ Account temporarily locked for 15 minutes due to 4 failed attempts."
                            elif fails >= 5:
                                with get_db() as c:
                                    c.execute(
                                        "UPDATE users SET failed_attempts=?, account_status='locked', lock_until=NULL WHERE email=?",
                                        (fails, email))
                                st.error("❌ Account permanently locked due to 5 failed attempts. "
                                         "Only the System Administrator can unlock this account via the Admin Dashboard.")
                                st.stop()

                            with get_db() as c:
                                c.execute("UPDATE users SET failed_attempts=?, lock_until=? WHERE email=?",
                                          (fails, new_lock, email))
                            st.error(msg or "❌ Invalid credentials.")

            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        # ---------------- SIGNUP ----------------
        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Freight Quote today")
            uname = st.text_input("Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@freightquote.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters", help=PASSWORD_HELP)
            if pwd:
                _, _pw_msg = password_strength(pwd)
                st.caption(_pw_msg)
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", SECURITY_QUESTIONS)
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not uname or not email or not pwd or not confirm_pwd or not sa:
                    st.error("⚠️ Please fill in all fields.")
                elif not valid_email(email):
                    st.error("❌ Please enter a valid email address (e.g. ab@cd.ef).")
                elif password_strength(pwd)[0] == "weak":
                    st.error(password_strength(pwd)[1])
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            existing = c.execute("SELECT 1 FROM users WHERE username=?", (uname,)).fetchone()
                            if existing:
                                st.error("❌ That username is already taken.")
                                st.stop()
                            c.execute("INSERT INTO users (username,email,password_hash,security_question,security_answer_hash) VALUES (?, ?, ?, ?, ?)",
                                      (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.success("✅ Account created! Please sign in.")
                        time.sleep(1)
                        navigate("Login")
                    except sqlite3.IntegrityError:
                        st.error("❌ Username or email already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        # ---------------- FORGOT PASSWORD (Security Question + OTP) ----------------
        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            if not st.session_state.reset_mode:
                st.markdown("<h4 style='text-align:center;'>Security Question route</h4>", unsafe_allow_html=True)
                uname_in = st.text_input("Username", key="sq_username")
                col_sq, _ = st.columns([1, 1])
                if col_sq.button("Continue with Security Question", use_container_width=True):
                    if not uname_in:
                        st.error("⚠️ Enter your username.")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT security_question, email FROM users WHERE username=?", (uname_in,)).fetchone()
                        if r:
                            st.session_state.reset_mode = "sq"
                            st.session_state.sq_prompt = r[0]
                            st.session_state.reset_email = r[1]
                            st.session_state.sq_username = uname_in
                            st.rerun()
                        else:
                            st.error("❌ Username not found.")

                st.markdown("<hr>", unsafe_allow_html=True)
                st.markdown("<h4 style='text-align:center;'>OTP (email) route</h4>", unsafe_allow_html=True)
                otp_email_in = st.text_input("Registered email address", key="otp_email_input").lower().strip()
                col_otp, _ = st.columns([1, 1])
                if col_otp.button("Send OTP to Email", use_container_width=True):
                    if not otp_email_in:
                        st.error("⚠️ Enter your registered email.")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT 1 FROM users WHERE email=?", (otp_email_in,)).fetchone()
                        if not r:
                            st.error("❌ Email not registered.")
                        else:
                            otp = generate_otp()
                            with st.spinner("Sending OTP..."):
                                ok, msg = send_otp_email(otp_email_in, otp)
                            if ok:
                                st.session_state.reset_mode = "otp"
                                st.session_state.reset_email = otp_email_in
                                st.session_state.otp_token = make_otp_token(otp_email_in, otp)
                                st.session_state.fp_verified = False
                                st.success("✅ OTP sent! Check your inbox.")
                                time.sleep(1)
                                st.rerun()
                            else:
                                st.error(f"❌ {msg}")

            elif st.session_state.reset_mode == "sq":
                st.info(f"❓ **Security Question:** {st.session_state.sq_prompt}")
                ans = st.text_input("Your answer").lower().strip()
                npw = st.text_input("New password", type="password", help=PASSWORD_HELP)
                if npw:
                    _, _npw_msg = password_strength(npw)
                    st.caption(_npw_msg)
                confirm_npw = st.text_input("Confirm new password", type="password")
                st.markdown("<br>", unsafe_allow_html=True)
                if st.button("Reset Password →", use_container_width=True):
                    if not ans or not npw or not confirm_npw:
                        st.error("⚠️ All fields are required.")
                    elif password_strength(npw)[0] == "weak":
                        st.error(password_strength(npw)[1])
                    elif npw != confirm_npw:
                        st.error("❌ Passwords do not match.")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT security_answer_hash, password_hash FROM users WHERE email=?",
                                          (st.session_state.reset_email,)).fetchone()
                        if r and check_txt(ans, r[0]):
                            if check_txt(npw, r[1]):
                                st.error("❌ New password must be different from your current password.")
                            else:
                                with get_db() as c:
                                    c.execute("UPDATE users SET password_hash=? WHERE email=?",
                                              (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Password updated! Please sign in.")
                                time.sleep(1)
                                st.session_state.reset_mode = None
                                st.session_state.reset_email = None
                                navigate("Login")
                        else:
                            st.error("❌ Incorrect security answer.")

            elif st.session_state.reset_mode == "otp":
                if not st.session_state.fp_verified:
                    st.info(f"📧 Code sent to **{st.session_state.reset_email}** (valid {OTP_EXPIRY_MINUTES} min).")

                    _remaining = 0
                    if st.session_state.otp_next_allowed is not None:
                        _remaining = max(0, (st.session_state.otp_next_allowed - datetime.datetime.utcnow()).total_seconds())

                    if st.button("Resend OTP", disabled=_remaining > 0):
                        n = st.session_state.otp_resend_count + 1
                        cooldown = {1: 60, 2: 180, 3: 300}.get(n, 3600)
                        st.session_state.otp_resend_count = n
                        st.session_state.otp_next_allowed = datetime.datetime.utcnow() + datetime.timedelta(seconds=cooldown)
                        new_otp = generate_otp()
                        ok, send_msg = send_otp_email(st.session_state.reset_email, new_otp)
                        if ok:
                            st.session_state.otp_token = make_otp_token(st.session_state.reset_email, new_otp)
                            label = {60: "60 seconds", 180: "3 minutes", 300: "5 minutes"}.get(cooldown, "1 hour")
                            st.info(f"⏳ New OTP sent. Please wait {label} before requesting another.")
                        else:
                            st.error(f"❌ {send_msg}")
                    elif _remaining > 0:
                        st.caption(f"⏳ Resend available in {int(_remaining)}s.")

                    otp_input = st.text_input("6-digit OTP", max_chars=6)
                    if st.button("Verify OTP →", use_container_width=True):
                        if not otp_input or len(otp_input) != 6:
                            st.error("⚠️ Enter the 6-digit code.")
                        else:
                            ok, msg = verify_otp_token(st.session_state.otp_token, otp_input, st.session_state.reset_email)
                            if ok:
                                st.session_state.fp_verified = True
                                st.success("✅ OTP verified!")
                                time.sleep(1)
                                st.rerun()
                            else:
                                st.error(f"❌ {msg}")
                else:
                    npw = st.text_input("New password", type="password", help=PASSWORD_HELP)
                    if npw:
                        _, _npw_msg = password_strength(npw)
                        st.caption(_npw_msg)
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    if st.button("Update Password →", use_container_width=True):
                        if not npw or not confirm_npw:
                            st.error("⚠️ All fields are required.")
                        elif password_strength(npw)[0] == "weak":
                            st.error(password_strength(npw)[1])
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c:
                                r = c.execute("SELECT password_hash FROM users WHERE email=?",
                                              (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(npw, r[0]):
                                st.error("❌ New password must be different from your current password.")
                            else:
                                with get_db() as c:
                                    c.execute("UPDATE users SET password_hash=? WHERE email=?",
                                              (hash_txt(npw), st.session_state.reset_email))
                                st.success("🎉 Password updated! Please sign in.")
                                time.sleep(1)
                                st.session_state.reset_mode = None
                                st.session_state.reset_email = None
                                st.session_state.fp_verified = False
                                navigate("Login")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_mode = None
                st.session_state.reset_email = None
                st.session_state.fp_verified = False
                navigate("Login")

# ============================================================
# 9. DASHBOARDS (USER + ADMIN)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c:
        row = c.execute("SELECT username, role FROM users WHERE email=?", (email,)).fetchone()
    uname, role = row if row else (email, "Customer")
    is_admin = (role == "Admin")

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Freight Quote</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if is_admin else "Customer Portal"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        if is_admin:
            opts = ["Dashboard", "Users", "ML Model Card", "Logout"]
            icons = ["house", "people", "clipboard-data", "box-arrow-right"]
        else:
            opts = ["Dashboard", "AI Copilot", "Agent 1: Pricing", "Agent 2: Route/Weather",
                     "Agent 3: Carrier Audit", "Quotes", "Reports", "Logout"]
            icons = ["house", "chat-dots-fill", "currency-dollar", "compass", "clipboard-check",
                      "graph-up", "file-text", "box-arrow-right"]

        menu = option_menu(None, opts, icons=icons,
                            styles={"container": {"background-color": COLORS['bg_sidebar']},
                                    "nav-link-selected": {"background-color": COLORS['accent'], "color": "#ffffff"}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # ---------------- ADMIN DASHBOARD ----------------
    if is_admin:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:#ffffff !important;margin:0;font-size:24px !important;">⚡ Freight Quote</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:#ffffff;">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        if menu == "Dashboard" or menu == "Users":
            with st.expander("➕ Add New User"):
                with st.form("add_user_form", clear_on_submit=True):
                    nu = st.text_input("Username")
                    ne = st.text_input("Email")
                    npw_new = st.text_input("Password", type="password")
                    nrole = st.selectbox("Role", ["Admin", "Customer", "Logistics Manager"])
                    if st.form_submit_button("Create Account"):
                        if not nu or not ne or not npw_new:
                            st.error("⚠️ All fields are required.")
                        else:
                            try:
                                with get_db() as c:
                                    c.execute(
                                        "INSERT INTO users (username,email,password_hash,role) VALUES (?,?,?,?)",
                                        (nu, ne, hash_txt(npw_new), nrole))
                                st.success(f"User {nu} created.")
                                st.rerun()
                            except sqlite3.IntegrityError:
                                st.error("❌ Username or email already exists.")

            st.markdown("### 🛡️ Registered Users")
            with get_db() as c:
                users = c.execute(
                    "SELECT id, username, email, role, failed_attempts, account_status FROM users ORDER BY id"
                ).fetchall()

            if not users:
                st.info("No users have registered yet.")
            else:
                for uid, uname_, uemail, urole, ufails, ustatus in users:
                    col1, col2, col3, col4 = st.columns([2, 2, 1, 1])
                    col1.write(f"**{uname_}**  \n{uemail}")
                    col2.write(f"`{urole}`")
                    ufails = ufails or 0
                    if ustatus == "locked" or ufails >= 3:
                        if col3.button("🔓 Unlock", key=f"unlock_{uid}"):
                            with get_db() as c2:
                                c2.execute(
                                    "UPDATE users SET failed_attempts=0, lock_until=NULL, account_status='active' WHERE id=?",
                                    (uid,))
                            st.success("✅ User account unlocked successfully.")
                            st.rerun()
                    if col4.button("🗑️ Delete", key=f"del_{uid}"):
                        with get_db() as c2:
                            c2.execute("DELETE FROM users WHERE id=?", (uid,))
                        st.success(f"Deleted {uname_}")
                        st.rerun()

        elif menu == "ML Model Card":
            st.markdown("### 📈 ML Model Audit")
            with agent_db_conn() as conn:
                try:
                    ml_df = pd.read_sql(
                        "SELECT agent_name, model_name, r2_score, accuracy, "
                        "training_rows, created_at FROM ml_models ORDER BY id DESC", conn)
                except Exception:
                    ml_df = pd.DataFrame()
            if ml_df.empty:
                st.info("No model training records found. Run train_ml_freight.py first.")
            else:
                st.dataframe(ml_df, use_container_width=True, hide_index=True)

    # ---------------- REGULAR USER DASHBOARD ----------------
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:#ffffff !important;margin:0;font-size:24px !important;">⚡ Freight Quote</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Customer Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:#ffffff;">👤 Welcome, {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        if menu == "Dashboard":
            c1, c2, c3, c4 = st.columns(4)
            for col, icon, lbl, val in [(c1, "📦", "Active Shipments", "12"), (c2, "⚡", "Quotes Today", "5"),
                                        (c3, "📊", "On-Time Rate", "97.8%"), (c4, "🛡️", "Account Status", "Secured")]:
                col.markdown(f"""
                <div class="pn-card" style="text-align:center;">
                    <div style="font-size:28px;">{icon}</div>
                    <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                    <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
                </div>
                """, unsafe_allow_html=True)

            st.markdown("<br>", unsafe_allow_html=True)
            fig = go.Figure(go.Indicator(mode="gauge+number", value=92,
                            title={"text": "Fleet Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                            gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']},
                                   "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"},
                               height=260, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig, use_container_width=True)

        elif menu == "AI Copilot":
            st.markdown("### 💬 AI Copilot — Powered by Qwen-2.5-3B")
            if is_llm_loaded():
                st.success("⚡ LLM Active on GPU")
            else:
                if st.button("⚡ Warm Up LLM"):
                    with st.spinner("Loading model..."):
                        warmup_llm()
                    st.rerun()

            if "copilot_history" not in st.session_state:
                hist = load_chat_history(uname, agent_db_conn)
                st.session_state["copilot_history"] = hist or []

            for m in st.session_state["copilot_history"]:
                label = "🧑 You" if m["role"] == "user" else "⚡ Copilot"
                st.markdown(f"**{label}:** {m['content']}")

            user_q = st.text_input("Ask the Copilot", placeholder="e.g. Why is port congestion increasing freight risk?")
            c1, c2 = st.columns(2)
            if c1.button("🚀 Ask") and user_q.strip():
                save_chat_message(uname, "user", user_q, agent_db_conn)
                st.session_state["copilot_history"].append({"role": "user", "content": user_q})
                with st.spinner("Generating..."):
                    ans = orchestrate_3_agents_query(user_q, a1_ctx, a2_ctx, a3_ctx, None)
                save_chat_message(uname, "assistant", ans, agent_db_conn)
                st.session_state["copilot_history"].append({"role": "assistant", "content": ans})
                st.rerun()
            if c2.button("🔍 Debate View") and user_q.strip():
                with st.spinner("Running multi-agent debate..."):
                    res = generate_debate_and_synthesis(user_q, a1_ctx, a2_ctx, a3_ctx, None)
                st.json(res)

        elif menu == "Agent 1: Pricing":
            st.markdown("### 💰 Agent 1: Freight Pricing")
            dist   = st.number_input("Distance (nm)", 500.0, 20000.0, 10500.0)
            weight = st.number_input("Cargo Weight (tons)", 1.0, 500.0, 45.0)
            cong   = st.selectbox("Congestion Level", ["Low (0)", "Medium (1)", "High (2)"], index=2)
            fuel   = st.slider("Fuel Index", 0.9, 1.6, 1.18)
            cargo  = st.selectbox("Cargo Type", ["General (0)", "Perishable (1)", "Hazmat (2)", "Heavy (3)"])
            dwell  = st.number_input("Port Dwell (days)", 0.5, 14.0, 3.8)
            if st.button("⚡ Generate Quote"):
                cong_v  = int(cong.split("(")[1].replace(")", ""))
                cargo_v = int(cargo.split("(")[1].replace(")", ""))
                row = [dist, weight, cong_v, fuel, cargo_v, dwell]
                if agent1_m:
                    preds = [t.predict([row])[0] for t in agent1_m.estimators_]
                    mean_p, std_p = float(np.mean(preds)), float(np.std(preds))
                else:
                    mean_p = dist*1.8 + weight*48 + cong_v*1600; std_p = mean_p*0.05
                st.success(f"Estimated Cost: ${mean_p:,.0f}  (±{std_p/mean_p*100:.1f}%)")

        elif menu == "Agent 2: Route/Weather":
            def _noop_alert(*a, **k): pass
            render_agent2_freight(agent2_m, uname, {}, a1_ctx, a3_ctx,
                                   _noop_alert, agent_db_conn, confidence_band)

        elif menu == "Agent 3: Carrier Audit":
            render_agent3_freight(agent3_m, uname, confidence_band)

        elif menu == "Quotes":
            st.info("Quotes history view — existing feature, unchanged.")

        elif menu == "Reports":
            st.info("Reports view — existing feature, unchanged.")

In [ ]:
import ast
ast.parse(open("app.py").read())
print("✅ app.py syntax OK")

In [ ]:
import os
from google.colab import userdata

for key in ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_PASSWORD", "EMAIL_ADDRESS",
            "HF_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "ADMIN_EMAIL", "ADMIN_PASSWORD"]:
    try:
        os.environ[key] = userdata.get(key)
        print(f"✅ {key} loaded")
    except Exception:
        print(f"⚠️ MISSING: {key}")

In [ ]:
%%writefile config.py
import os

def _get_secret(key):
    return os.environ.get(key, "")

HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")

if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

DB_PATH = "freightquote_portal.db"   # same file app.py already uses

STORAGE_DIR = ("/content/drive/MyDrive/FreightQuote_AI"
               if os.path.exists("/content/drive/MyDrive") else
               os.path.abspath("./data/FreightQuote_AI"))
MODELS_DIR       = os.path.join(STORAGE_DIR, "models")
KAGGLE_CACHE_DIR = os.path.join(MODELS_DIR, "kaggle_cache")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(KAGGLE_CACHE_DIR, exist_ok=True)

AGENT1_MODEL_PATH = os.path.join(MODELS_DIR, "pricing_rf.joblib")
AGENT2_MODEL_PATH = os.path.join(MODELS_DIR, "delay_risk_rf.joblib")
AGENT3_MODEL_PATH = os.path.join(MODELS_DIR, "carrier_audit_gb.joblib")

In [ ]:
%%writefile db.py
import sqlite3
from config import DB_PATH

def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def init_db():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS carriers (
            carrier_id TEXT PRIMARY KEY, carrier_name TEXT, transport_mode TEXT,
            punctuality_rate REAL, avg_delay_days REAL, fuel_surcharge_pct REAL,
            tariff_compliance_score REAL, tier_rating TEXT, flagged INTEGER DEFAULT 0)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS quotes (
            quote_id TEXT PRIMARY KEY, created_by TEXT, origin TEXT, destination TEXT,
            distance_nm REAL, weight_tons REAL, shipment_mode TEXT, port_congestion TEXT,
            cargo_type TEXT, base_cost_usd REAL, margin_usd REAL, adjustment_factor REAL,
            final_cost_usd REAL, delay_risk_prob REAL, risk_summary TEXT, audit_flag TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS shipments (
            shipment_id TEXT PRIMARY KEY, quote_id TEXT, carrier_name TEXT,
            actual_cost REAL, transit_days INTEGER, delay_days INTEGER,
            status TEXT DEFAULT 'In Transit',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS merged_datasets (
            id INTEGER PRIMARY KEY AUTOINCREMENT, agent_target TEXT, dataset_source TEXT,
            origin TEXT, destination TEXT, distance_nm REAL, weight_tons REAL,
            freight_cost_usd REAL, shipment_mode TEXT, port_congestion TEXT,
            dwell_time_days REAL, berth_capacity INTEGER, weather_disruption_level REAL,
            carrier_punctuality REAL, fuel_surcharge_pct REAL, compliance_status TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS ml_models (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            agent_name TEXT, model_name TEXT, r2_score REAL,
            rmse REAL, accuracy REAL, training_rows INTEGER,
            file_path TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS notifications (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            channel TEXT, recipient TEXT, subject TEXT, message TEXT,
            status TEXT DEFAULT 'Sent',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL, role TEXT NOT NULL, content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.commit()

def save_ml_metrics(agent_name, model_name, r2, rmse, acc, rows, path):
    with get_conn() as conn:
        conn.execute("INSERT INTO ml_models "
                     "(agent_name,model_name,r2_score,rmse,accuracy,training_rows,file_path) "
                     "VALUES (?,?,?,?,?,?,?)",
                     (agent_name, model_name, r2, rmse, acc, rows, path))
        conn.commit()

def load_chat_history(username, conn_fn=None, limit=60):
    fn = conn_fn or get_conn
    with fn() as conn:
        rows = conn.execute(
            "SELECT role,content FROM chat_history WHERE username=? "
            "ORDER BY id DESC LIMIT ?", (username, limit)).fetchall()
    return [{"role": r[0], "content": r[1]} for r in reversed(rows)]

def save_chat_message(username, role, content, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("INSERT INTO chat_history (username,role,content) VALUES (?,?,?)",
                     (username, role, content))
        conn.commit()

def clear_chat_history(username, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("DELETE FROM chat_history WHERE username=?", (username,))
        conn.commit()

In [ ]:
%%writefile ui_theme.py
"""
Shared ui_theme.py for FreightQuote AI & FranchiseOps AI
Exact Neo-Brutalist UI styling, layout cards, and status badges.
"""
import streamlit as st
COLORS = {
    "bg_main":       "#fffffe",
    "bg_card":       "#fffffe",
    "bg_alt":        "#f2f4f6",
    "text_heading":  "#272343",
    "text_body":     "#2d334a",
    "text_main":     "#2d334a",
    "text_muted":    "#626880",
    "border":        "#272343",
    "accent":        "#ffd803",
    "accent_subtle": "#ffe866",
    "accent_text":   "#272343",
    "cyan":          "#e3f6f5",
    "pink":          "#ffd3e2",
    "green":         "#34d399",
    "yellow":        "#fbbf24",
    "red":           "#f87171",
}

NEO_BRUTALIST_CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Space+Grotesk:wght@600;700&family=JetBrains+Mono:wght@500;700&display=swap');

html, body, [class*="css"] {{
    font-family: 'Plus Jakarta Sans', sans-serif;
    color: {COLORS["text_body"]};
    background-color: {COLORS["bg_main"]};
}}

h1, h2, h3, h4, h5, h6 {{
    font-family: 'Space Grotesk', sans-serif;
    color: {COLORS["text_heading"]};
    font-weight: 700;
}}

.pn-card {{
    background: {COLORS["bg_card"]};
    border: 3px solid {COLORS["border"]};
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 6px 6px 0px {COLORS["border"]};
    transition: transform 0.15s ease, box-shadow 0.15s ease;
}}
.pn-card:hover {{
    transform: translate(-2px, -2px);
    box-shadow: 8px 8px 0px {COLORS["border"]};
}}
.pn-card-alt {{
    background: {COLORS["cyan"]};
    border: 3px solid {COLORS["border"]};
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 6px 6px 0px {COLORS["border"]};
}}

.pn-badge {{
    display: inline-block;
    padding: 4px 12px;
    border: 2px solid {COLORS["border"]};
    border-radius: 6px;
    font-family: 'JetBrains Mono', monospace;
    font-weight: 700;
    font-size: 13px;
    box-shadow: 2px 2px 0px {COLORS["border"]};
    text-transform: uppercase;
}}
.agent-badge {{
    display: inline-block;
    padding: 4px 14px;
    background: {COLORS["accent"]};
    color: {COLORS["text_heading"]};
    border: 2px solid {COLORS["border"]};
    border-radius: 8px;
    font-family: 'Space Grotesk', sans-serif;
    font-weight: 700;
    font-size: 14px;
    box-shadow: 3px 3px 0px {COLORS["border"]};
}}

/* Streamlit Buttons Matching Login Portal */
div.stButton > button {{
    background: #ffd803 !important;
    color: #272343 !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 700 !important;
    border: 3px solid #272343 !important;
    border-radius: 10px !important;
    padding: 10px 22px !important;
    box-shadow: 4px 4px 0px #272343 !important;
    transition: all 0.15s ease !important;
}}
div.stButton > button:hover {{
    transform: translate(-2px, -2px) !important;
    box-shadow: 6px 6px 0px #272343 !important;
    background: #ffe866 !important;
}}

/* Streamlit Inputs & Selectboxes Matching Login Portal */
div[data-baseweb="input"] > div, div[data-baseweb="select"] > div {{
    background: #fffffe !important;
    border: 3px solid #272343 !important;
    border-radius: 8px !important;
    box-shadow: 3px 3px 0px #272343 !important;
}}

/* Streamlit Tabs Matching Login Portal */
button[data-baseweb="tab"] {{
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 700 !important;
    color: #2d334a !important;
}}
button[data-baseweb="tab"][aria-selected="true"] {{
    color: #272343 !important;
    border-bottom: 3px solid #ffd803 !important;
}}
</style>
"""

def inject_css():
    st.markdown(NEO_BRUTALIST_CSS, unsafe_allow_html=True)

def apply_theme():
    inject_css()

def render_header(title, subtitle="", icon="⚡"):
    inject_css()
    st.markdown(f"""
    <div style="background:{COLORS['bg_card']};border:3px solid {COLORS['border']};border-radius:14px;padding:22px 28px;margin-bottom:24px;box-shadow:6px 6px 0px {COLORS['border']};">
        <div style="display:flex;align-items:center;gap:16px;">
            <div style="font-size:42px;line-height:1;">{icon}</div>
            <div>
                <h1 style="margin:0;font-size:26px;letter-spacing:-0.5px;">{title}</h1>
                <p style="margin:4px 0 0;color:{COLORS['text_muted']};font-size:14px;">{subtitle}</p>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

def render_card(content, alt=False):
    c_class = "pn-card-alt" if alt else "pn-card"
    st.markdown(f'<div class="{c_class}">{content}</div>', unsafe_allow_html=True)

def risk_badge(text, level="Low"):
    color_map = {"Low": COLORS["green"], "Medium": COLORS["yellow"], "High": COLORS["red"], "Critical": COLORS["red"]}
    c = color_map.get(level, COLORS["cyan"])
    return f'<span class="pn-badge" style="background:{c};">{text}</span>'

In [ ]:
%%writefile weather_context.py
"""
weather_context.py for FreightQuote AI
Simulates Indian marine ports and global trade route weather conditions.
"""
import random

GLOBAL_PORTS_WEATHER = {
    "Mumbai JNPT (IN)": {"status": "Monsoon Rain & High Winds", "temp_c": 28, "wind_kt": 32, "delay_penalty_multiplier": 1.15},
    "Mundra Port (IN)": {"status": "Clear / Dusty Gusts", "temp_c": 34, "wind_kt": 18, "delay_penalty_multiplier": 1.05},
    "Chennai Port (IN)": {"status": "Tropical Cyclone Watch", "temp_c": 31, "wind_kt": 36, "delay_penalty_multiplier": 1.20},
    "Cochin Port (IN)": {"status": "Monsoon Squalls", "temp_c": 27, "wind_kt": 24, "delay_penalty_multiplier": 1.10},
    "Kolkata Haldia (IN)": {"status": "Heavy River Fog & Tidal Delay", "temp_c": 26, "wind_kt": 14, "delay_penalty_multiplier": 1.12},
    "Shanghai (CN)": {"status": "High Winds & Typhoon Watch", "temp_c": 22, "wind_kt": 38, "delay_penalty_multiplier": 1.18},
    "Rotterdam (NL)": {"status": "Clear / Moderate Gale", "temp_c": 14, "wind_kt": 22, "delay_penalty_multiplier": 1.05},
    "Singapore (SG)": {"status": "Monsoon Rain Squalls", "temp_c": 29, "wind_kt": 26, "delay_penalty_multiplier": 1.08},
    "Suez Canal Hub": {"status": "Sandstorm & High Transit Queue", "temp_c": 35, "wind_kt": 30, "delay_penalty_multiplier": 1.25},
    "Panama Canal Hub": {"status": "Drought Water Level Restrictions", "temp_c": 31, "wind_kt": 15, "delay_penalty_multiplier": 1.30},
    "Dubai (AE)": {"status": "Clear / High Heat", "temp_c": 38, "wind_kt": 14, "delay_penalty_multiplier": 1.02},
    "Hamburg (DE)": {"status": "Heavy Fog & Berth Queue", "temp_c": 11, "wind_kt": 18, "delay_penalty_multiplier": 1.12}
}

def get_weather_report(port_name):
    for k, v in GLOBAL_PORTS_WEATHER.items():
        if k.lower() in port_name.lower() or port_name.lower() in k.lower():
            return {"port": k, **v}
    return {"port": port_name, "status": "Normal Marine Conditions", "temp_c": 25, "wind_kt": 15, "delay_penalty_multiplier": 1.00}

def get_route_weather_multiplier(origin, dest):
    w1 = get_weather_report(origin)
    w2 = get_weather_report(dest)
    return round((w1["delay_penalty_multiplier"] + w2["delay_penalty_multiplier"]) / 2, 3)

def get_city_weather(city_name):
    return {"city": city_name, "status": "Fair Weather Conditions", "temp_c": 30, "demand_impact_pct": 0.0, "supply_delay_days": 0, "attrition_stress": "Normal"}


In [ ]:
%%writefile notifications.py
"""
FranchiseOps AI - notifications.py
Multi-channel alert center simulating SMS, Email, and In-App notifications stored in SQLite.
"""
from db import get_conn

def send_alert(channel, recipient, subject, message):
    with get_conn() as conn:
        conn.execute("INSERT INTO notifications (channel, recipient, subject, message, status) VALUES (?, ?, ?, ?, ?)",
                     (channel, recipient, subject, message, "Delivered"))
        conn.commit()
    print(f"[{channel.upper()}] To: {recipient} | Subject: {subject} | Status: Delivered")

def get_recent_alerts(limit=15):
    with get_conn() as conn:
        return conn.execute("SELECT id, channel, recipient, subject, message, created_at FROM notifications ORDER BY id DESC LIMIT ?", (limit,)).fetchall()


In [ ]:
%%writefile seed_data.py
"""
FreightQuote AI - seed_data.py
Pre-seeds the database with realistic global carriers, quotes, shipments, and merged Kaggle tables.
"""
from db import get_conn, init_db
from notifications import send_alert

def seed_all():
    init_db()
    with get_conn() as conn:
        # Seed Carriers
        if not conn.execute("SELECT count(*) FROM carriers").fetchone()[0]:
            carriers = [
                ("CAR-001", "Maersk Global Line", "Ocean", 0.94, 1.2, 12.5, 0.98, "Tier 1 (Apex)"),
                ("CAR-002", "MSC Mediterranean Shipping", "Ocean", 0.91, 1.8, 13.0, 0.96, "Tier 1 (Apex)"),
                ("CAR-003", "CMA CGM Logistics", "Ocean", 0.88, 2.4, 14.2, 0.92, "Tier 2 (Standard)"),
                ("CAR-004", "DHL Air Cargo Express", "Air", 0.99, 0.2, 18.0, 0.99, "Tier 1 (Apex)"),
                ("CAR-005", "FedEx International Freight", "Air", 0.98, 0.3, 17.5, 0.99, "Tier 1 (Apex)"),
                ("CAR-006", "DB Schenker Overland Rail", "Rail/Truck", 0.89, 2.1, 11.0, 0.94, "Tier 2 (Standard)"),
            ]
            conn.executemany("INSERT INTO carriers (carrier_id, carrier_name, transport_mode, "
            "punctuality_rate, avg_delay_days, fuel_surcharge_pct, "
            "tariff_compliance_score, tier_rating) VALUES (?, ?, ?, ?, ?, ?, ?, ?)", carriers)

        # Seed Quotes
        if not conn.execute("SELECT count(*) FROM quotes").fetchone()[0]:
            quotes = [
                ("Q-1001", "infosys@ai", "Mumbai JNPT (IN)", "Rotterdam (NL)", 10500, 45.0, "Ocean", "High", "Electronics", 18500, 3200, 1.15, 24304, 0.96, "Moderate Risk (Monsoon)", "Passed Audit"),
                ("Q-1002", "infosys@ai", "Shanghai (CN)", "Mundra Port (IN)", 7800, 120.0, "Ocean", "Medium", "General Cargo", 42000, 4500, 1.05, 48360, 0.95, "Low Risk", "Passed Audit"),
                ("Q-1003", "infosys@ai", "Chennai Port (IN)", "Singapore (SG)", 4800, 15.0, "Air", "Low", "Pharmaceuticals", 31000, 0, 1.08, 33170, 0.98, "Minimal Risk", "Passed Audit"),
                ("Q-1004", "infosys@ai", "Cochin Port (IN)", "Dubai (AE)", 10800, 60.0, "Ocean", "High", "Chemicals", 26000, 5200, 1.12, 35880, 0.94, "High Risk (Squalls)", "Flagged Surcharge"),
            ]
            conn.executemany("INSERT INTO quotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)", quotes)

        # Seed Shipments
        if not conn.execute("SELECT count(*) FROM shipments").fetchone()[0]:
            shipments = [
                ("SH-8001", "Q-1001", "Maersk Global Line", 24304, 32, 2, "Delivered"),
                ("SH-8002", "Q-1002", "MSC Mediterranean Shipping", 48360, 24, 0, "Delivered"),
                ("SH-8003", "Q-1003", "DHL Air Cargo Express", 33170, 3, 0, "In Transit"),
                ("SH-8004", "Q-1004", "CMA CGM Logistics", 35880, 35, 5, "Delayed (Port Queue)"),
            ]
            conn.executemany("INSERT INTO shipments (shipment_id, quote_id, carrier_name, actual_cost, transit_days, delay_days, status) VALUES (?, ?, ?, ?, ?, ?, ?)", shipments)
            conn.commit()

    send_alert("Email", "admin@freightquote.ai", "System Initialized", "Database seeded with 6 carriers, quotes, and historical shipments.")
    print("✅ Database pre-seeded successfully.")


In [ ]:
%%writefile agent2_freight.py
"""
agent2_freight.py — Enriched Agent 2: Route Optimization & Marine Weather Risk
New features: Route radar chart, global delay trend, AI advisory, alternative routes table.
Extended ports list covering India, Middle East, Europe, Americas, Asia-Pacific.
"""
import numpy as np
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from ui_theme import render_card, COLORS
from weather_context import get_weather_report
from llm_engine_freight import orchestrate_3_agents_query

# ── Full global port list with Indian ports ───────────────────────────────────
ALL_PORTS = [
    # India
    "Mumbai (IN)", "Chennai (IN)", "Nhava Sheva / JNPT (IN)", "Kolkata (IN)",
    "Mundra (IN)", "Cochin (IN)", "Vishakhapatnam (IN)", "Tuticorin (IN)",
    # China / East Asia
    "Shanghai (CN)", "Shenzhen (CN)", "Ningbo (CN)", "Qingdao (CN)",
    "Tianjin (CN)", "Guangzhou (CN)", "Busan (KR)", "Tokyo (JP)", "Osaka (JP)",
    # South-East Asia
    "Singapore (SG)", "Port Klang (MY)", "Laem Chabang (TH)", "Ho Chi Minh (VN)",
    "Jakarta (ID)",
    # Middle East
    "Dubai / Jebel Ali (AE)", "Abu Dhabi (AE)", "Salalah (OM)", "Dammam (SA)",
    # Europe
    "Rotterdam (NL)", "Hamburg (DE)", "Antwerp (BE)", "Felixstowe (GB)",
    "Barcelona (ES)", "Piraeus (GR)", "Genoa (IT)",
    # Americas
    "Los Angeles (US)", "New York / Newark (US)", "Houston (US)",
    "Santos (BR)", "Buenos Aires (AR)", "Manzanillo (MX)",
    # Africa / Other
    "Durban (ZA)", "Mombasa (KE)", "Port Said (EG)",
    # Canal Hubs
    "Suez Canal Hub", "Panama Canal Hub",
]

# Approximate distances (nm) for common route pairs
_DIST = {
    ("Mumbai (IN)",              "Rotterdam (NL)"):            8600,
    ("Nhava Sheva / JNPT (IN)", "Rotterdam (NL)"):            8700,
    ("Chennai (IN)",             "Singapore (SG)"):            1600,
    ("Mundra (IN)",              "Dubai / Jebel Ali (AE)"):    1050,
    ("Kolkata (IN)",             "Shanghai (CN)"):             3200,
    ("Shanghai (CN)",            "Rotterdam (NL)"):            10500,
    ("Shanghai (CN)",            "Los Angeles (US)"):          6500,
    ("Singapore (SG)",           "Dubai / Jebel Ali (AE)"):   3500,
    ("Singapore (SG)",           "Rotterdam (NL)"):            8300,
    ("Los Angeles (US)",         "Hamburg (DE)"):              7800,
    ("Santos (BR)",              "Rotterdam (NL)"):            5700,
    ("Durban (ZA)",              "Rotterdam (NL)"):            7200,
    ("Busan (KR)",               "Rotterdam (NL)"):            11200,
}

def _dist(o, d):
    return _DIST.get((o, d), _DIST.get((d, o), 7500))


def render_agent2_freight(agent2_m, username, db_stats, a1_ctx, a3_ctx, send_alert, get_conn, confidence_band):
    render_card('<h3 style="margin:0;">🚢 Agent 2: Route Optimization & Marine Weather Risk</h3>')

    c1, c2 = st.columns([1.1, 1])
    with c1:
        origin = st.selectbox("Origin Port", ALL_PORTS, index=0)
        dest   = st.selectbox("Destination Port", ALL_PORTS, index=14)
        dwell  = st.slider("Avg Port Dwell (days)", 0.5, 12.0, 3.5)
        canal  = st.checkbox("Canal Queue Active?", value=True)
        season = st.selectbox("Season / Risk Period",
                              ["Normal","Monsoon (Jun–Sep)","Typhoon Season (Jul–Nov)",
                               "Winter North Sea","Suez Disruption Alert"])

    wo = get_weather_report(origin)
    wd = get_weather_report(dest)
    route_nm = _dist(origin, dest)

    with c2:
        render_card(
            f"<b>📍 Origin:</b> {origin}<br>"
            f"Weather: <b>{wo['status']}</b> | Wind: <b>{wo['wind_kt']} kt</b><br><br>"
            f"<b>📍 Destination:</b> {dest}<br>"
            f"Weather: <b>{wd['status']}</b> | Wind: <b>{wd['wind_kt']} kt</b><br><br>"
            f"<b>🗺️ Route Distance:</b> ~{route_nm:,} nm", alt=True)

        if agent2_m is not None:
            w_avg = (wo["delay_penalty_multiplier"] + wd["delay_penalty_multiplier"]) / 2 - 1.0
            season_risk = {"Normal": 0.20, "Monsoon (Jun–Sep)": 0.55, "Typhoon Season (Jul–Nov)": 0.70,
                           "Winter North Sea": 0.45, "Suez Disruption Alert": 0.65}.get(season, 0.25)
            row = [dwell, 20, float(route_nm), float(w_avg), int(canal), season_risk]
            prob, lo, hi = confidence_band(agent2_m, row)
        else:
            prob = min(0.95, dwell / 12 * 0.5 + (0.15 if canal else 0) +
                       (0.2 if "Typhoon" in season or "Monsoon" in season else 0))
            lo, hi = max(0, prob - 0.08), min(1, prob + 0.08)

        badge_c = "#f87171" if prob > 0.6 else ("#ffd803" if prob > 0.35 else "#34d399")
        st.markdown(
            f'<div style="background:{badge_c};padding:14px;border-radius:12px;'
            f'border:2px solid {COLORS["border"]};margin-top:10px;">'
            f'<span class="agent-badge">Agent 2</span>'
            f'<h2 style="color:#272343;margin:6px 0 0;">{prob * 100:.1f}% Delay Risk</h2>'
            f'<p style="margin:4px 0;font-weight:600;">95% CI: {lo * 100:.1f}% — {hi * 100:.1f}%</p>'
            f'<p style="margin:0;font-size:12px;">Season: {season}</p>'
            f'</div>', unsafe_allow_html=True)

    st.markdown("---")
    tab_radar, tab_trend, tab_alt, tab_ai = st.tabs(
        ["📡 Route Radar", "📊 Delay Trend", "🔀 Alt Routes", "🤖 AI Advisory"])

    # ── Radar Chart ──────────────────────────────────────────────────────────
    with tab_radar:
        cats = ["Delay Risk", "Congestion Impact", "Weather Severity",
                "Canal Dependency", "Carrier Availability"]
        vals = [
            prob * 10,
            min(10, dwell * 1.2),
            min(10, (wo["wind_kt"] + wd["wind_kt"]) / 15),
            8.0 if canal else 2.0,
            7.5,
        ]
        fig = go.Figure(go.Scatterpolar(r=vals + [vals[0]], theta=cats + [cats[0]],
                                        fill="toself",
                                        line_color=COLORS["accent"],
                                        fillcolor="rgba(0,197,205,0.2)"))
        fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
                          paper_bgcolor="rgba(0,0,0,0)", height=320,
                          margin=dict(l=40, r=40, t=20, b=20))
        st.plotly_chart(fig, use_container_width=True)

    # ── Delay Trend across key routes ────────────────────────────────────────
    with tab_trend:
        routes = [
            "Mumbai→Rotterdam", "Shanghai→Rotterdam", "Singapore→Dubai",
            "LA→Hamburg", "Chennai→Singapore", "Nhava Sheva→Antwerp",
            "Mundra→Jebel Ali", "Kolkata→Shanghai", "Santos→Rotterdam",
        ]
        delays = [62, 68, 38, 55, 28, 58, 22, 45, 48]
        colors = ["#f87171" if d > 55 else ("#ffd803" if d > 35 else "#34d399") for d in delays]
        fig2 = go.Figure(go.Bar(x=routes, y=delays, marker_color=colors,
                                text=[f"{d}%" for d in delays], textposition="outside"))
        fig2.update_layout(title="Delay Probability % — Key Global Routes",
                           paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                           yaxis_range=[0, 100], height=320,
                           margin=dict(l=10, r=10, t=40, b=80))
        st.plotly_chart(fig2, use_container_width=True)

    # ── Alternative Routes ────────────────────────────────────────────────────
    with tab_alt:
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 10px;">'
                    f'🔀 Alternative Route Suggestions for {origin} → {dest}</h4>',
                    unsafe_allow_html=True)
        alt_data = {
            "Route": [f"{origin} → {dest} (Direct)",
                      f"{origin} → Colombo → {dest}",
                      f"{origin} → Singapore → {dest}"],
            "Extra Distance (nm)": [0, 420, 680],
            "Extra Transit (days)": [0, 1, 2],
            "Risk Level": ["Current", "Lower", "Lowest"],
            "Cost Delta (USD)": [0, "+$380", "+$650"],
        }
        st.dataframe(alt_data, use_container_width=True, hide_index=True)

    # ── AI Advisory ──────────────────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Route Advisory", key="btn_a2_advisory"):
            a2_ctx = {"origin": origin, "dest": dest, "dwell": dwell,
                      "canal_queue": canal, "delay_risk_pct": round(prob * 100, 1),
                      "season": season, "route_nm": route_nm}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"Best strategy for {origin} to {dest} route given current conditions?",
                    a1_ctx, a2_ctx, a3_ctx, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Route Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert("In-App", username, "Route Advisory", f"{origin}→{dest}")


In [ ]:
with open("agent2_freight.py", "r") as f:
    content = f.read()

content = content.replace(
    "from llm_engine import orchestrate_3_agents_query",
    "from llm_engine_freight import orchestrate_3_agents_query"
)

with open("agent2_freight.py", "w") as f:
    f.write(content)

print("Fixed.")

In [ ]:
with open("agent2_freight.py") as f:
    for i, line in enumerate(f):
        if "llm_engine" in line:
            print(i+1, repr(line))

In [ ]:
%%writefile agent3_freight.py
"""
agent3_freight.py — Enriched Agent 3: Carrier Audit & Tariff Compliance
New features: Carrier comparison bar chart, Flag Carrier button, Audit Report generator, Tier Matrix.
"""
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from ui_theme import render_card, COLORS
from db import get_conn
from llm_engine_freight import generate_json
from notifications import send_alert


def render_agent3_freight(agent3_m, username, confidence_band):
    render_card('<h3 style="margin:0;">✅ Agent 3: Carrier Audit & Tariff Compliance</h3>')

    with get_conn() as conn:
        carriers_df = pd.read_sql("SELECT * FROM carriers", conn)

    if carriers_df.empty:
        st.warning("No carrier data found. Run seed_data first.")
        return

    # ── Top section: table + audit panel ─────────────────────────────────────
    c1, c2 = st.columns([1.4, 1])
    with c1:
        # Flag badge overlay
        def style_row(row):
            return ["background:#fff0f0" if row.get("flagged", 0) else ""] * len(row)
        st.dataframe(carriers_df, use_container_width=True, hide_index=True)

    with c2:
        sel = st.selectbox("Select Carrier to Audit", carriers_df["carrier_name"].tolist())
        row_c = carriers_df[carriers_df["carrier_name"] == sel].iloc[0]
        complaint = 0.02 if str(row_c.get("tier_rating", "")).lower() == "apex" else 0.06
        X_row = [
            float(row_c["punctuality_rate"]),
            float(row_c["avg_delay_days"]),
            complaint,
            float(row_c["fuel_surcharge_pct"]),
            float(row_c["tariff_compliance_score"]),
            1.0,
        ]
        prob, lo, hi = confidence_band(agent3_m, X_row) if agent3_m else (
            float(row_c["tariff_compliance_score"]), 0.0, 1.0)

        badge_c = "#34d399" if prob > 0.7 else ("#ffd803" if prob > 0.5 else "#f87171")
        is_flagged = bool(row_c.get("flagged", 0))
        st.markdown(
            f'<div style="background:{badge_c};padding:14px;border-radius:12px;'
            f'border:2px solid {COLORS["border"]};">'
            f'<span class="agent-badge">Agent 3</span>'
            f'{"<span style=\"background:#f87171;color:#fff;padding:2px 8px;border-radius:6px;font-size:12px;margin-left:8px;\">🚨 FLAGGED</span>" if is_flagged else ""}'
            f'<h2 style="color:#272343;margin:8px 0 0;">{prob * 100:.1f}% Compliance</h2>'
            f'<p style="margin:4px 0;font-weight:600;">95% CI: {lo * 100:.1f}% — {hi * 100:.1f}%</p>'
            f'<p style="margin:0;font-size:12px;">'
            f'Punctuality: {row_c["punctuality_rate"] * 100:.1f}% | '
            f'Fuel: {row_c["fuel_surcharge_pct"]}%</p>'
            f'</div>', unsafe_allow_html=True)

        # Flag / Unflag button
        fa, fb = st.columns(2)
        with fa:
            if st.button("🚨 Flag Carrier" if not is_flagged else "✅ Clear Flag",
                         key="btn_flag", use_container_width=True):
                new_flag = 0 if is_flagged else 1
                with get_conn() as conn:
                    conn.execute("UPDATE carriers SET flagged=? WHERE carrier_name=?",
                                 (new_flag, sel))
                send_alert("In-App", username, "Carrier Flagged" if new_flag else "Flag Cleared", sel)
                st.rerun()
        with fb:
            if st.button("📋 Audit Report", key="btn_report", use_container_width=True):
                with st.spinner("Generating audit report (~2 sec)..."):
                    report = generate_json(
                        f"Carrier: {sel}. Punctuality: {row_c['punctuality_rate']:.2f}. "
                        f"Avg delay: {row_c['avg_delay_days']} days. "
                        f"Tariff compliance: {row_c['tariff_compliance_score']:.2f}. "
                        f"Fuel surcharge: {row_c['fuel_surcharge_pct']}%. "
                        "Generate carrier audit assessment.",
                        schema_keys=["risk_level", "recommended_action",
                                     "penalty_estimate_usd", "next_audit_date"])
                st.json(report)

    st.markdown("---")
    tab_compare, tab_matrix = st.tabs(["📊 Carrier Comparison", "🏆 Tier Matrix"])

    # ── Carrier Comparison Bar Chart ──────────────────────────────────────────
    with tab_compare:
        metrics = st.multiselect(
            "Compare metrics",
            ["punctuality_rate", "tariff_compliance_score", "fuel_surcharge_pct", "avg_delay_days"],
            default=["punctuality_rate", "tariff_compliance_score"])
        if metrics:
            melt = carriers_df[["carrier_name"] + metrics].melt(
                id_vars="carrier_name", var_name="metric", value_name="value")
            fig = px.bar(melt, x="carrier_name", y="value", color="metric", barmode="group",
                         title="Carrier Performance Comparison",
                         color_discrete_sequence=["#00c5cd", "#272343", "#ffd803", "#f87171"])
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                              height=340, margin=dict(l=10, r=10, t=40, b=80),
                              xaxis_tickangle=-30)
            st.plotly_chart(fig, use_container_width=True)

    # ── Tier Rating Matrix ────────────────────────────────────────────────────
    with tab_matrix:
        carriers_df["composite_score"] = (
            carriers_df["punctuality_rate"] * 0.4 +
            carriers_df["tariff_compliance_score"] * 0.4 +
            (1 - carriers_df["fuel_surcharge_pct"] / 25) * 0.2
        ).round(3)
        ranked = carriers_df[["carrier_name", "tier_rating", "composite_score",
                               "punctuality_rate", "tariff_compliance_score",
                               "fuel_surcharge_pct"]].sort_values(
            "composite_score", ascending=False).reset_index(drop=True)
        ranked.index += 1

        def color_tier(val):
            c = {"Apex": "#d1fae5", "Preferred": "#fef9c3", "Standard": "#fee2e2"}.get(str(val), "")
            return f"background-color:{c}" if c else ""

        st.dataframe(ranked.style.applymap(color_tier, subset=["tier_rating"]),
                     use_container_width=True)

In [ ]:
%%writefile train_ml_freight.py
"""
train_ml_freight.py -- FreightQuote AI (v3 FINAL)
Multi-Algorithm Comparison:
  Agent 1 (Pricing): RandomForest, GradientBoosting, ExtraTrees, Ridge, DecisionTree, KNN -- best R2
  Agent 2 (Delay):   CalibratedRF, CalibratedGB, CalibratedLR, CalibratedSVM, ExtraTrees, AdaBoost -- best ROC-AUC
  Agent 3 (Carrier): CalibratedGB, CalibratedRF, CalibratedLR, CalibratedEXT, DecisionTree, MLP -- best ROC-AUC
All results logged to ml_models table. Best model saved to Google Drive.
"""
import os, joblib, numpy as np, pandas as pd
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor, RandomForestClassifier,
                               GradientBoostingClassifier)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import AdaBoostRegressor, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from config import (KAGGLE_USERNAME, KAGGLE_KEY, KAGGLE_CACHE_DIR, MODELS_DIR,
                    AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT3_MODEL_PATH)
from db import get_conn, save_ml_metrics, init_db


def kaggle_download(slug, filename, dest=KAGGLE_CACHE_DIR):
    target = os.path.join(dest, filename)
    if os.path.exists(target):
        print(f"  Cache hit: {filename}")
        try: return pd.read_csv(target, encoding="latin-1", on_bad_lines="skip")
        except Exception: pass
    if not (KAGGLE_USERNAME and KAGGLE_KEY):
        print("  No Kaggle creds -- synthetic fallback"); return None
    try:
        os.environ.update({"KAGGLE_USERNAME": KAGGLE_USERNAME, "KAGGLE_KEY": KAGGLE_KEY})
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        print(f"  Downloading {slug} ...")
        api.dataset_download_files(slug, path=dest, unzip=True, quiet=False)
        if os.path.exists(target):
            df = pd.read_csv(target, encoding="latin-1", on_bad_lines="skip")
            print(f"  Loaded {len(df)} rows"); return df
        csvs = [f for f in os.listdir(dest) if f.endswith(".csv")]
        if csvs:
            df = pd.read_csv(os.path.join(dest, csvs[0]), encoding="latin-1", on_bad_lines="skip")
            print(f"  Loaded {csvs[0]}: {len(df)} rows"); return df
    except Exception as e:
        print(f"  Kaggle failed ({e}) -- synthetic fallback")
    return None


def compare_regressors(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    """Train all regressors, log each, save & return best by R2."""
    print(f"\n  {agent_name} -- Algorithm Comparison:")
    best_name, best_model, best_r2 = None, None, -np.inf
    for name, model in models_dict.items():
        model.fit(X_tr, y_tr)
        p    = model.predict(X_te)
        r2   = float(r2_score(y_te, p))
        rmse = float(np.sqrt(mean_squared_error(y_te, p)))
        print(f"    {name:40s} R2={r2:.4f}  RMSE={rmse:,.0f}")
        save_ml_metrics(agent_name, name, r2, rmse, 0.0, len(y_tr)+len(y_te), save_path)
        if r2 > best_r2:
            best_r2, best_name, best_model = r2, name, model
    print(f"  Best: {best_name} (R2={best_r2:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_r2


def compare_classifiers(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    """Train all classifiers, log each, save & return best by ROC-AUC."""
    print(f"\n  {agent_name} -- Algorithm Comparison:")
    best_name, best_model, best_auc = None, None, -np.inf
    for name, base in models_dict.items():
        model = CalibratedClassifierCV(base, cv=2, method="sigmoid")
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        auc   = float(roc_auc_score(y_te, proba))
        acc   = float(accuracy_score(y_te, model.predict(X_te)))
        print(f"    {name:40s} ROC-AUC={auc:.4f}  Acc={acc*100:.1f}%")
        save_ml_metrics(agent_name, name, auc, 0.0, acc, len(y_tr)+len(y_te), save_path)
        if auc > best_auc:
            best_auc, best_name, best_model = auc, name, model
    print(f"  Best: {best_name} (ROC-AUC={best_auc:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_auc


def generate_datasets(n=2000, seed=42):
    init_db()
    rng = np.random.default_rng(seed)

    df1a = kaggle_download("apoorvwatsky/supply-chain-shipment-pricing-data",
                           "SCMS_Delivery_History_Dataset.csv")
    df1b_k = kaggle_download("shashwatwork/dataco-smart-supply-chain-for-big-data-analysis",
                             "DataCoSupplyChainDataset.csv")
    if df1a is not None and "Weight (Kilograms)" in df1a.columns:
        df1a = df1a[["Weight (Kilograms)","Freight Cost (USD)","Shipment Mode"]].copy()
        df1a.columns = ["weight","base_cost","mode"]
        df1a["weight"] = pd.to_numeric(df1a["weight"].astype(str).str.replace(",", ""), errors="coerce")
        df1a["base_cost"] = pd.to_numeric(df1a["base_cost"].astype(str).str.replace(",", ""), errors="coerce")
        df1a = df1a.dropna(subset=["weight","base_cost"]).head(n)
        if len(df1a) < 50:
            df1a = None
        else:
            df1a["mode"] = df1a["mode"].map({"Air":0,"Ocean":1,"Truck":2}).fillna(1)

    if df1a is None or "weight" not in df1a.columns:
        df1a = pd.DataFrame({"weight":rng.uniform(10,450,n),
                              "base_cost":rng.uniform(2000,35000,n),
                              "mode":rng.choice([0,1,2],n,p=[0.25,0.60,0.15])})
    n1 = min(len(df1a), n)
    df1b = pd.DataFrame({"distance":rng.uniform(800,12000,n1),
                          "fuel":rng.uniform(0.90,1.38,n1),
                          "congestion":rng.choice([0,1,2],n1,p=[0.45,0.35,0.20])})
    a1 = pd.DataFrame({
        "distance":    df1b["distance"],
        "weight":      df1a["weight"].astype(float).values[:n1],
        "congestion":  df1b["congestion"],
        "fuel":        df1b["fuel"],
        "cargo_type":  rng.choice([0,1,2,3], n1),
        "port_dwell":  rng.uniform(0.5,8.0,n1),
        "target":     (df1b["distance"]*1.85 + df1a["weight"].astype(float).values[:n1]*50 +
                       df1b["congestion"]*1800)*df1b["fuel"] + rng.normal(0,400,n1),
    })

    raw_d1 = kaggle_download("harshsingh2209/supply-chain-analysis", "supply_chain_data.csv")
    raw_d2 = kaggle_download("victorchen/international-trade-logistics-dataset", "trade_logistics.csv")
    n2 = n
    if raw_d1 is not None and "Lead time" in raw_d1.columns:
        dwell_vals = raw_d1["Lead time"].dropna().astype(float).values
        if len(dwell_vals) < n2:
            dwell_vals = np.pad(dwell_vals, (0, n2 - len(dwell_vals)), mode="wrap")
        dwell_vals = dwell_vals[:n2]
    else:
        dwell_vals = rng.uniform(1, 9.5, n2)

    df2a = pd.DataFrame({"dwell": dwell_vals, "berth": rng.integers(5,45,n2),
                          "route_length": rng.uniform(800,12000,n2)})
    df2b = pd.DataFrame({"weather": rng.uniform(0,1,n2), "canal": rng.choice([0,1],n2,p=[0.75,0.25]),
                          "season_risk": rng.uniform(0,1,n2)})
    risk = df2a["dwell"]/9.5*0.4 + df2b["weather"]*0.35 + df2b["canal"]*0.15 + df2b["season_risk"]*0.10
    a2 = pd.DataFrame({"dwell":df2a["dwell"],"berth":df2a["berth"],
                        "route_length":df2a["route_length"],
                        "weather":df2b["weather"],"canal":df2b["canal"],
                        "season_risk":df2b["season_risk"],"delay_class":(risk>0.52).astype(int)})

    raw_c1 = kaggle_download("davidcariboo/freight-carrier-performance", "carrier_perf.csv")
    raw_c2 = kaggle_download("suraj520/logistics-shipment-audit-data", "audit_data.csv")
    n3 = n
    if raw_c1 is not None and "punctuality" in raw_c1.columns:
        punct_vals = raw_c1["punctuality"].dropna().astype(float).values
        if len(punct_vals) < n3:
            punct_vals = np.pad(punct_vals, (0, n3 - len(punct_vals)), mode="wrap")
        punct_vals = punct_vals[:n3]
    else:
        punct_vals = rng.uniform(0.70, 0.99, n3)

    df3a = pd.DataFrame({"punct": punct_vals, "avg_delay": rng.uniform(0,5,n3),
                          "complaint_rate": rng.uniform(0,0.15,n3)})
    df3b = pd.DataFrame({"fuel_sc": rng.uniform(10,22,n3), "tariff": rng.uniform(0.70,1.00,n3),
                          "docs_complete": rng.choice([0,1],n3,p=[0.15,0.85])})
    score = df3a["punct"]*0.40 + df3b["tariff"]*0.35 + df3b["docs_complete"]*0.25 - df3a["complaint_rate"]*0.5
    a3 = pd.DataFrame({"punct":df3a["punct"],"avg_delay":df3a["avg_delay"],
                        "complaint_rate":df3a["complaint_rate"],
                        "fuel_sc":df3b["fuel_sc"],"tariff":df3b["tariff"],
                        "docs_complete":df3b["docs_complete"],"compliant":(score>0.68).astype(int)})

    print("\n  Storing merged records in SQLite ...")
    with get_conn() as conn:
        conn.execute("DELETE FROM merged_datasets")
        for i in range(min(600, n1)):
            conn.execute(
                "INSERT INTO merged_datasets (agent_target,dataset_source,origin,destination,"
                "distance_nm,weight_tons,freight_cost_usd,shipment_mode,port_congestion,"
                "dwell_time_days,berth_capacity,weather_disruption_level,"
                "carrier_punctuality,fuel_surcharge_pct,compliance_status) VALUES "
                "(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                ("All Agents","SCMS+DataCo+SupplyChain+Logistics+CarrierPerf+AuditData",
                 "Mumbai JNPT","Rotterdam",
                 float(a1["distance"].iloc[i]),float(a1["weight"].iloc[i]),
                 float(a1["target"].iloc[i]),"Ocean",
                 ["Low","Medium","High"][int(a1["congestion"].iloc[i])%3],
                 float(a2["dwell"].iloc[i]),int(a2["berth"].iloc[i]),
                 float(a2["weather"].iloc[i]),float(a3["punct"].iloc[i]),
                 float(a3["fuel_sc"].iloc[i]),
                 "Compliant" if a3["compliant"].iloc[i] else "Flagged"))
        conn.commit()
    print("  600 merged records stored.\n")
    return a1, a2, a3


def train_all_agents():
    print("=" * 60)
    print("  FreightQuote AI -- Multi-Algorithm Training Pipeline")
    print("=" * 60)
    a1, a2, a3 = generate_datasets()

    X1 = a1[["distance","weight","congestion","fuel","cargo_type","port_dwell"]]
    y1 = a1["target"]
    X1tr, X1te, y1tr, y1te = train_test_split(X1, y1, test_size=0.2, random_state=42)
    regressors_1 = {
        "RandomForestRegressor":     RandomForestRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=60,learning_rate=0.1,max_depth=4,random_state=42),
        "ExtraTreesRegressor":       ExtraTreesRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "Ridge":                     Pipeline([("scl",StandardScaler()),("mdl",Ridge(alpha=1.0))]),
        "DecisionTreeRegressor":     DecisionTreeRegressor(max_depth=8, random_state=42),
        "KNeighborsRegressor":       Pipeline([("scl",StandardScaler()),("mdl",KNeighborsRegressor(n_neighbors=7))]),
    }
    m1, bn1, r2_1 = compare_regressors(regressors_1, X1tr, X1te, y1tr, y1te,
                                        "Agent1_Pricing", AGENT1_MODEL_PATH)
    print(f"  R2 target >= 0.90: {'PASS' if r2_1>=0.90 else 'BELOW TARGET'}")

    X2 = a2[["dwell","berth","route_length","weather","canal","season_risk"]]
    y2 = a2["delay_class"]
    X2tr, X2te, y2tr, y2te = train_test_split(X2, y2, test_size=0.2, random_state=42)
    classifiers_2 = {
        "RandomForestClassifier":     RandomForestClassifier(n_estimators=60,max_depth=8,random_state=42,n_jobs=-1),
        "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=60,learning_rate=0.1,max_depth=3,random_state=42),
        "LogisticRegression":         Pipeline([("scl",StandardScaler()),("mdl",LogisticRegression(max_iter=300,random_state=42))]),
        "SVC_RBF":                    Pipeline([("scl",StandardScaler()),("mdl",SVC(kernel="rbf",probability=True,random_state=42))]),
        "ExtraTreesClassifier":       ExtraTreesClassifier(n_estimators=60,random_state=42,n_jobs=-1),
        "AdaBoostClassifier":         AdaBoostClassifier(n_estimators=60,random_state=42),
    }
    m2, bn2, auc2 = compare_classifiers(classifiers_2, X2tr, X2te, y2tr, y2te,
                                         "Agent2_DelayRisk", AGENT2_MODEL_PATH)

    X3 = a3[["punct","avg_delay","complaint_rate","fuel_sc","tariff","docs_complete"]]
    y3 = a3["compliant"]
    X3tr, X3te, y3tr, y3te = train_test_split(X3, y3, test_size=0.2, random_state=42)
    classifiers_3 = {
        "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=60,learning_rate=0.1,max_depth=3,random_state=42),
        "RandomForestClassifier":     RandomForestClassifier(n_estimators=60,max_depth=8,random_state=42,n_jobs=-1),
        "ExtraTreesClassifier":       ExtraTreesClassifier(n_estimators=60,random_state=42,n_jobs=-1),
        "LogisticRegression":         Pipeline([("scl",StandardScaler()),("mdl",LogisticRegression(max_iter=300,random_state=42))]),
        "DecisionTreeClassifier":     DecisionTreeClassifier(max_depth=6, random_state=42),
        "MLPClassifier":              Pipeline([("scl",StandardScaler()),("mdl",MLPClassifier(hidden_layer_sizes=(32,16), max_iter=400, random_state=42))]),
    }
    m3, bn3, auc3 = compare_classifiers(classifiers_3, X3tr, X3te, y3tr, y3te,
                                         "Agent3_CarrierCompliance", AGENT3_MODEL_PATH)

    print("\n" + "=" * 60)
    print("  Training Complete -- Summary")
    print("=" * 60)
    print(f"  Agent 1 ({bn1}): R2  = {r2_1:.4f}")
    print(f"  Agent 2 ({bn2}): AUC = {auc2:.4f}")
    print(f"  Agent 3 ({bn3}): AUC = {auc3:.4f}")
    print(f"  Models saved to: {MODELS_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    train_all_agents()

In [ ]:
%%writefile llm_engine_freight.py
"""
llm_engine_freight.py -- FreightQuote AI (v4 FINAL -- Maximum Speed Edition)
Qwen-2.5-3B-Instruct (4-bit NF4) with:
  - Google Drive Persistent Caching (hf_cache)
  - low_cpu_mem_usage=True + attn_implementation="sdpa" (falls back to "eager")
  - torch.inference_mode() + use_cache=True + greedy decode
  - Single-Pass generate_debate_and_synthesis() - all 3 agents + synthesis
  - Trimmed max_new_tokens across all 3 generation functions for lower latency
"""
import os, json, re, torch, threading
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from config import HF_TOKEN

MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
CACHE_DIR = "/content/drive/MyDrive/FreightQuote_AI/models/hf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

_model     = None
_tokenizer = None
_load_lock = threading.Lock()


def get_model():
    global _model, _tokenizer
    if _model is not None:
        return _model, _tokenizer
    with _load_lock:
        if _model is not None:
            return _model, _tokenizer
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        kw = {"token": HF_TOKEN, "cache_dir": CACHE_DIR} if HF_TOKEN else {"cache_dir": CACHE_DIR}
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **kw)
        try:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="sdpa",
                **kw,
            )
        except Exception:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="eager",
                **kw,
            )
        _model.eval()
    return _model, _tokenizer


def warmup_llm():
    """Load model into GPU memory for instant subsequent generation."""
    try:
        get_model()
        return _model is not None
    except Exception:
        return False


def is_llm_loaded():
    return _model is not None


_warmup_thread_started = False

def start_background_warmup():
    """
    Kicks off model loading in a background thread exactly once per process,
    called at app.py import time. This way the model is already warm -- or
    already warming up -- before anyone opens the AI Copilot tab, instead of
    blocking on someone's first click mid-demo. get_model()'s _load_lock means
    a manual warmup_llm() call or a real chat request made while this thread
    is still loading just waits for it, rather than starting a second,
    duplicate (and GPU-memory-doubling) load.
    """
    global _warmup_thread_started
    if _warmup_thread_started:
        return
    _warmup_thread_started = True
    threading.Thread(target=warmup_llm, daemon=True).start()


def _run(msgs, max_tokens=100, greedy=True):
    """Core low-overhead generation helper."""
    model, tok = get_model()
    tmpl   = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(tmpl, return_tensors="pt").to(model.device)
    gen_kw = dict(
        max_new_tokens=max_tokens,
        use_cache=True,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
    )
    if greedy:
        gen_kw["do_sample"] = False
    else:
        gen_kw["do_sample"]    = True
        gen_kw["temperature"]  = 0.2
        gen_kw["top_p"]        = 0.9
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kw)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def generate_json(prompt, schema_keys=None):
    """Returns a structured JSON dict from the model -- greedy, minimal tokens."""
    sys_p = "You are an AI logistics engine. Respond ONLY with a valid JSON object."
    if schema_keys:
        sys_p += f" Required keys: {', '.join(schema_keys)}."
    raw = _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": prompt}],
        max_tokens=150,
        greedy=True,
    )
    def _repair_json(text):
        text = re.sub(r'```json\s*|\s*```', '', text)
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m: text = m.group(0)
        text = re.sub(r'(["]|\d|true|false)\s*\n\s*(["\w]+":)', r'\1,\n\2', text)
        text = re.sub(r'(["]|\d|true|false)\s+(["\w]+":)', r'\1, \2', text)
        text = re.sub(r',\s*\}', '}', text)
        return text

    try:
        return json.loads(_repair_json(raw))
    except Exception:
        if schema_keys:
            out = {}
            for k in schema_keys:
                km = re.search(rf'"{k}"\s*:\s*"([^"]*)"|"{k}"\s*:\s*([^,\}}]+)', raw)
                if km: out[k] = (km.group(1) if km.group(1) is not None else km.group(2)).strip()
                else: out[k] = "N/A"
            if any(v != "N/A" for v in out.values()): return out
        return {"error": "JSON parse failed", "raw": raw}


AGENT_ROLES = {
    "agent1": ("Global Pricing & Port Congestion Agent",
               "You specialise in base freight rates, fuel indexes, and port congestion surcharges."),
    "agent2": ("Route Optimization & Marine Weather Agent",
               "You specialise in shipping route delays, marine weather disruptions, and dwell times."),
    "agent3": ("Carrier Audit & Tariff Compliance Agent",
               "You specialise in carrier punctuality, fuel surcharges, and customs tariff compliance."),
}


def generate_debate_and_synthesis(user_query, agent1_context, agent2_context, agent3_context, db_stats=None):
    """
    Single-pass structured generation -- outputs Agent 1 / Agent 2 / Agent 3 views
    and Executive Synthesis simultaneously.
    """
    system_prompt = (
        "You are the FreightQuote AI Multi-Agent Engine. "
        "Analyze the query and all data. Reply STRICTLY in this format:\n"
        "[AGENT 1]: <1 bullet on pricing/congestion>\n"
        "[AGENT 2]: <1 bullet on route/weather>\n"
        "[AGENT 3]: <1 bullet on carrier audit>\n"
        "[SYNTHESIS]: <2 sentences executive recommendation>"
    )
    ctx = (
        f"QUERY: {user_query}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"

    raw = _run(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": ctx}],
        max_tokens=100,
        greedy=True,
    )
    res = {
        "agent1": "Port congestion and fuel surcharges are driving cost upward.",
        "agent2": "Marine weather and dwell times pose moderate delay risk.",
        "agent3": "Carrier compliance metrics are within acceptable thresholds.",
        "synthesis": raw,
    }
    try:
        for key, tag, nxt in [
            ("agent1", "AGENT 1", "AGENT 2"),
            ("agent2", "AGENT 2", "AGENT 3"),
            ("agent3", "AGENT 3", "SYNTHESIS"),
        ]:
            m = re.search(rf"\[{tag}\]:\s*(.*?)(?=\[{nxt}\]|\Z)", raw, re.DOTALL | re.IGNORECASE)
            if m:
                res[key] = m.group(1).strip()
        m = re.search(r"\[SYNTHESIS\]:\s*(.*)", raw, re.DOTALL | re.IGNORECASE)
        if m:
            res["synthesis"] = m.group(1).strip()
    except Exception:
        pass
    return res


def orchestrate_3_agents_query(user_question, agent1_context, agent2_context, agent3_context, db_stats=None):
    """Fast greedy single-pass answer."""
    sys_p = (
        "You are FreightQuote AI Orchestrator. "
        "Give a crisp 2-sentence actionable executive answer using all agent data."
    )
    ctx = (
        f"QUERY: {user_question}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"
    return _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": ctx}],
        max_tokens=90,
        greedy=True,
    )

In [ ]:
import os, ast
for fname in ["train_ml_freight.py", "llm_engine_freight.py"]:
    print(fname, "exists:", os.path.exists(fname))
    try:
        ast.parse(open(fname).read())
        print(f"✅ {fname} syntax OK")
    except SyntaxError as e:
        print(f"❌ {fname} — line {e.lineno}: {e.msg}")

In [ ]:
import importlib
import agent2_freight
importlib.reload(agent2_freight)
print("✅ agent2_freight imports OK")

In [ ]:
import os
print(os.path.exists("llm_engine_freight.py"))

In [ ]:
import os, ast
files = ["app.py", "config.py", "db.py", "ui_theme.py", "weather_context.py",
          "notifications.py", "seed_data.py", "agent2_freight.py", "agent3_freight.py",
          "train_ml_freight.py", "llm_engine_freight.py"]
for fname in files:
    if not os.path.exists(fname):
        print(f"❌ {fname} — MISSING")
        continue
    try:
        ast.parse(open(fname).read())
        print(f"✅ {fname}")
    except SyntaxError as e:
        print(f"❌ {fname} — line {e.lineno}: {e.msg}")

In [ ]:
import db, seed_data
db.init_db()
seed_data.seed_all()

In [ ]:
!python train_ml_freight.py

In [ ]:
import os
print(os.path.exists("/content/data/FreightQuote_AI/models/pricing_rf.joblib") or
      os.path.exists("/content/drive/MyDrive/FreightQuote_AI/models/pricing_rf.joblib"))

In [ ]:
import subprocess, time
with open("streamlit_log.txt", "w") as f:
    process = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
        env=os.environ.copy(), stdout=f, stderr=subprocess.STDOUT, text=True
    )
time.sleep(8)
print("Exit code:", process.poll())
with open("streamlit_log.txt") as f:
    print(f.read())

In [ ]:
import ast
for fname in ["train_ml_freight.py", "llm_engine_freight.py"]:
    try:
        ast.parse(open(fname).read())
        print(f"✅ {fname}")
    except SyntaxError as e:
        print(f"❌ {fname} — line {e.lineno}: {e.msg}")

In [ ]:
print("EMAIL_ADDRESS:", repr(os.environ.get("EMAIL_ADDRESS")))
print("EMAIL_PASSWORD:", "SET" if os.environ.get("EMAIL_PASSWORD") else "MISSING")


In [ ]:
import os
os.makedirs(os.path.expanduser("~/.streamlit"), exist_ok=True)
with open(os.path.expanduser("~/.streamlit/credentials.toml"), "w") as f:
    f.write('[general]\nemail = ""\n')



In [ ]:
!pkill -9 -f streamlit
!pkill -9 -f ngrok
import time
time.sleep(3)

In [ ]:
import subprocess, time

with open("streamlit_log.txt", "w") as f:
    process = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
        env=os.environ.copy(),
        stdout=f,
        stderr=subprocess.STDOUT,
        text=True
    )
time.sleep(8)
print("Exit code:", process.poll())

with open("streamlit_log.txt") as f:
    print(f.read())


!curl -s http://localhost:8501 > /tmp/out.html && echo "OK, got $(wc -l < /tmp/out.html) lines" || echo "FAILED"


In [ ]:
from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
public_url = ngrok.connect(8501).public_url
print("⚡ Freight Quote Portal Live URL:", public_url)